# Exp49 — PRR-B100: candidate-budget-matched retrieval control

**Kernel:** `Python 3.12 (Stock Regime)`

This experiment addresses the natural fairness question:

> Full PRR reranks about 183 unique candidates on average, whereas the fixed Pearson support contains 100.  
> Does PRR still improve retrieval when its **inference-time candidate budget is capped at 100**?

## Definition

We use the **same trained full-PRR checkpoints** as the submitted model and change **only the
inference candidate support**:

- Pearson branch: Top-50
- learned-representation branch: Top-50
- union: at most 100 unique candidates
- final reranking: the same frozen full-PRR fusion reranker
- final selection: Top-10
- lookback \(L=96\), memory stride 8, same frozen query manifests, same temporal admissibility

We call this **PRR-B100**.

This is intentionally an **inference-budget ablation, not a retrained model**. Holding the learned
encoders and reranker fixed isolates the effect of reducing the test-time candidate support from
`Top100 ∪ Top100` (about 183 unique candidates) to `Top50 ∪ Top50` (≤100 unique candidates).

The experiment should be interpreted together with PRR-Stat:

- **PRR-Stat:** same Pearson Top-100 support, future-supervised reranking only.
- **PRR-B100:** learned + Pearson support, but ≤100 total inference candidates.
- **Full PRR:** learned + Pearson support with Top100 ∪ Top100, ~183 unique candidates.

If PRR-B100 remains clearly stronger than Pearson, then Full PRR's gain cannot be explained
simply by exposing the reranker to more than 100 candidates at inference.


In [1]:
from pathlib import Path
import os, sys, re, json, time, gc, math, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])
print("PyTorch          :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())

if sys.version_info < (3, 12):
    raise RuntimeError("Select the Python 3.12 (Stock Regime) kernel.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for this experiment.")

ROOT_CANDIDATES = [
    Path(os.environ.get("PRR_ROOT", "")) if os.environ.get("PRR_ROOT") else None,
    Path("/data/code/which_histories_matter_ext"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster"),
]
ROOT = next(
    (p for p in ROOT_CANDIDATES if p is not None and
     (p / "results" / "exp33" / "final_eval_manifest").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find PRR project root. Set PRR_ROOT or place the notebook "
        "under the existing which_histories_matter_ext project."
    )

EXP33 = ROOT / "results" / "exp33"
EXP34 = ROOT / "results" / "exp34"
MANIFEST_ROOT = EXP33 / "final_eval_manifest"
DIRECT_ROOT = EXP34 / "patchtst_direct_cache"

RESULT_ROOT = ROOT / "results" / "exp49_prr_budget100"
PART_ROOT = RESULT_ROOT / "channel_parts"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
PART_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)

CHANNELS = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

DEVICE = torch.device("cuda:0")
print("GPU              :", torch.cuda.get_device_name(0))
print("ROOT             :", ROOT)
print("RESULT_ROOT      :", RESULT_ROOT)

class CFG:
    retrieval_seq_len = 96
    memory_stride = 8
    top_k = 10

    # Candidate-budget control:
    top_m_pattern = 50
    top_m_learned = 50
    union_max = 100

    emb_dim = 64
    enc_hidden = 128
    rerank_hidden = 384
    rerank_dropout = 0.1
    feature_clip = 8.0
    seed = 0
    query_batch = 64

cfg = CFG()
EPS = 1e-8

assert cfg.top_m_pattern + cfg.top_m_learned == 100
assert cfg.union_max == 100

print("[PASS] PRR-B100 configuration: Top50 Pearson + Top50 learned -> union <= 100")


Python executable: /data/envs/stock_regime/bin/python
Python version   : 3.12.13
PyTorch          : 2.12.1+cu126
CUDA available   : True
GPU              : NVIDIA A100-SXM4-80GB
ROOT             : /data/code/which_histories_matter_ext
RESULT_ROOT      : /data/code/which_histories_matter_ext/results/exp49_prr_budget100
[PASS] PRR-B100 configuration: Top50 Pearson + Top50 learned -> union <= 100


In [2]:
# Cell 2 — resolve the 24 PRR full-stack paths

REVIEWER_STACK_ROOT = ROOT / "artifacts" / "full_prr_training" / "full_stacks"

def legacy_stack_path(dataset, horizon):
    if dataset == "ETTh1":
        return (
            ROOT
            / "checkpoints"
            / "11_etth1_all_horizons_five_backbones_fixed_alpha01"
            / "full_stacks"
            / f"ETTh1_H{int(horizon)}_Full.pt"
        )
    return (
        ROOT
        / "checkpoints"
        / "12_multidataset_screening_itransformer_segmoe_fixed_alpha01"
        / "full_stacks"
        / f"{dataset}_H{int(horizon)}_screen.pt"
    )

def reviewer_stack_path(dataset, horizon):
    suffix = "Full" if dataset == "ETTh1" else "screen"
    return REVIEWER_STACK_ROOT / f"{dataset}_H{int(horizon)}_{suffix}.pt"

STACKS = {}
rows = []

for d in DATASETS:
    for h in HORIZONS:
        reviewer = reviewer_stack_path(d, h)
        legacy = legacy_stack_path(d, h)

        if reviewer.is_file():
            chosen = reviewer
            source = "reviewer_training"
        elif legacy.is_file():
            chosen = legacy
            source = "legacy_experiment"
        else:
            chosen = reviewer
            source = "missing"

        STACKS[(d, h)] = chosen
        rows.append({
            "dataset": d,
            "horizon": h,
            "exists": chosen.exists(),
            "source": source,
            "size_MB": chosen.stat().st_size / 2**20 if chosen.exists() else np.nan,
            "path": str(chosen),
        })

stack_inventory = pd.DataFrame(rows)
display(stack_inventory)

assert len(stack_inventory) == 24
if not stack_inventory.exists.all():
    missing = stack_inventory[~stack_inventory.exists]
    raise FileNotFoundError(
        "Missing PRR full-stack checkpoints. Run 01_full_prr_training.ipynb "
        "or set PRR_ROOT to a repository containing the original checkpoints.\n"
        + missing.to_string(index=False)
    )

print("[PASS] PRR full-stack coverage = 24/24")

,dataset,horizon,exists,source,size_MB,path
0,ETTh1,96,True,legacy_experiment,1.756768,/data/code/which_histories_matter_ext/checkpoi...
1,ETTh1,192,True,legacy_experiment,1.756800,/data/code/which_histories_matter_ext/checkpoi...
2,ETTh1,336,True,legacy_experiment,1.756800,/data/code/which_histories_matter_ext/checkpoi...
3,ETTh1,720,True,legacy_experiment,1.756800,/data/code/which_histories_matter_ext/checkpoi...
4,Weather,96,True,legacy_experiment,1.759885,/data/code/which_histories_matter_ext/checkpoi...
5,Weather,192,True,legacy_experiment,1.759916,/data/code/which_histories_matter_ext/checkpoi...
6,Weather,336,True,legacy_experiment,1.759916,/data/code/which_histories_matter_ext/checkpoi...
7,Weather,720,True,legacy_experiment,1.759916,/data/code/which_histories_matter_ext/checkpoi...
8,Electricity,96,True,legacy_experiment,1.758790,/data/code/which_histories_matter_ext/checkpoi...
9,Electricity,192,True,legacy_experiment,1.758821,/data/code/which_histories_matter_ext/checkpoi...


[PASS] PRR full-stack coverage = 24/24


In [3]:

# Cell 3 — raw dataset paths and train-only normalization

DATA_CANDIDATES = {
    "ETTh1": [
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
    ],
    "Weather": [
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
    ],
    "Electricity": [
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
    ],
    "Traffic": [
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
    ],
    "Exchange": [
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
    ],
    "Solar": [
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
    ],
}

def resolve_data(dataset):
    p=next((p for p in DATA_CANDIDATES[dataset] if p.exists()),None)
    if p is None:
        raise FileNotFoundError(dataset)
    return p

def read_raw(dataset,path):
    if dataset=="Solar":
        try:
            df=pd.read_csv(path,header=None)
            if df.shape[1]==1:
                df=pd.read_csv(path,header=None,sep=r"\s+")
        except Exception:
            df=pd.read_csv(path,header=None,sep=r"\s+")
    else:
        df=pd.read_csv(path)
        if "date" in df.columns:
            df=df.drop(columns=["date"])
        if "OT" in df.columns:
            df=df[[x for x in df.columns if x!="OT"]+["OT"]]

    df=(
        df.apply(pd.to_numeric,errors="coerce")
        .dropna(axis=1,how="all")
        .replace([np.inf,-np.inf],np.nan)
        .interpolate(axis=0,limit_direction="both")
        .ffill()
        .bfill()
    )
    return df.to_numpy(dtype=np.float32)

def bounds(dataset,n):
    if dataset=="ETTh1":
        train_end=12*30*24
        val_end=train_end+4*30*24
        test_end=n
    else:
        train_end=int(.70*n)
        val_end=n-int(.20*n)
        test_end=n
    return train_end,val_end,test_end

DATA={}

for d in DATASETS:
    p=resolve_data(d)
    raw=read_raw(d,p)
    assert raw.shape[1]==CHANNELS[d], (d,raw.shape)

    train_end,val_end,test_end=bounds(d,len(raw))
    mu=raw[:train_end].mean(axis=0).astype(np.float32)
    sd=raw[:train_end].std(axis=0,ddof=0).astype(np.float32)
    assert np.all(sd>1e-6)

    z=((raw-mu[None,:])/sd[None,:]).astype(np.float32)

    DATA[d]={
        "path":p,
        "raw":raw,
        "z":z,
        "train_end":train_end,
        "val_end":val_end,
        "test_end":test_end,
    }

    print(
        f"{d:11s}: rows={len(raw):6d} C={raw.shape[1]:4d} "
        f"train={train_end} val={val_end}"
    )

print("[PASS] all six datasets normalized from training only")


ETTh1      : rows= 17420 C=   7 train=8640 val=11520
Weather    : rows= 52696 C=  21 train=36887 val=42157
Electricity: rows= 26304 C= 321 train=18412 val=21044
Traffic    : rows= 17544 C= 862 train=12280 val=14036
Exchange   : rows=  7588 C=   8 train=5311 val=6071
Solar      : rows= 52560 C= 137 train=36792 val=42048
[PASS] all six datasets normalized from training only


In [4]:

# Cell 4 — exact PRR architecture and helper functions

def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def batch_pattern(x):
    x=np.asarray(x,np.float32)
    xc=x-x.mean(axis=-1,keepdims=True)
    n=np.linalg.norm(xc,axis=-1,keepdims=True)
    return np.where(
        n>EPS,
        xc/np.maximum(n,EPS),
        0.0
    ).astype(np.float32)

def context7(x):
    x=np.asarray(x,np.float32)
    short=max(8,cfg.retrieval_seq_len//4)
    m=x.mean(axis=-1)
    s=x.std(axis=-1)+EPS

    f1=(x[...,-1]-m)/s
    f2=(x[...,-short:].mean(axis=-1)-m)/s
    f3=(x[...,-1]-x[...,-short])/s
    f4=(x[...,-1]-x[...,0])/s

    df=np.diff(x,axis=-1)
    ds=np.diff(x[...,-short:],axis=-1)
    f5=(ds.std(axis=-1)+EPS)/(df.std(axis=-1)+EPS)

    t=np.linspace(-1.,1.,cfg.retrieval_seq_len,dtype=np.float32)
    t=t-t.mean()
    f6=(
        np.sum(t*(x-m[...,None]),axis=-1)
        /(np.sum(t*t)+EPS)
    )/s

    a=x[...,:-1]
    b=x[...,1:]
    a=a-a.mean(axis=-1,keepdims=True)
    b=b-b.mean(axis=-1,keepdims=True)
    f7=(
        np.sum(a*b,axis=-1)
        /(
            np.sqrt(np.sum(a*a,axis=-1)*np.sum(b*b,axis=-1))
            +EPS
        )
    )

    return np.stack([f1,f2,f3,f4,f5,f6,f7],axis=-1).astype(np.float32)

class TemporalEncoder1D(nn.Module):
    def __init__(self,hidden=128,emb_dim=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv1d(1,hidden//2,5,stride=2,padding=2),
            nn.GELU(),
            nn.Conv1d(hidden//2,hidden,5,stride=2,padding=2),
            nn.GELU(),
            nn.Conv1d(hidden,hidden,3,stride=2,padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj=nn.Linear(hidden,emb_dim)

    def forward(self,x):
        if x.ndim==2:
            x=x[:,None,:]
        h=self.net(x).squeeze(-1)
        return F.normalize(self.proj(h),dim=-1)

def inverse_softplus(x):
    return math.log(math.exp(float(x))-1.)

class WideFusionReranker(nn.Module):
    def __init__(self):
        super().__init__()
        in_dim=1+4*(7+cfg.emb_dim)
        h=cfg.rerank_hidden
        self.net=nn.Sequential(
            nn.Linear(in_dim,h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h,h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h,1),
        )
        self.raw_alpha=nn.Parameter(
            torch.tensor(inverse_softplus(.1),dtype=torch.float32)
        )

    def forward(self,feat,base,mask):
        delta=self.net(feat).squeeze(-1)
        alpha=F.softplus(self.raw_alpha)
        score=base+alpha*delta
        return score.masked_fill(~mask,-1e9)

@torch.no_grad()
def encode_rows(enc,x,batch=512):
    out=[]
    enc.eval()
    for left in range(0,len(x),batch):
        t=torch.from_numpy(x[left:left+batch]).float().to(DEVICE)
        out.append(enc(t).cpu().numpy())
    return np.concatenate(out,axis=0).astype(np.float32)

class UpgradedStack:
    def __init__(self,qenc,henc,reranker,stats_med,stats_iqr,
                 dataset,horizon,training_channels,
                 training_query_anchors,rerank_best_epoch):
        self.qenc=qenc.eval()
        self.henc=henc.eval()
        self.reranker=reranker.eval()
        self.stats_med=np.asarray(stats_med,np.float32)
        self.stats_iqr=np.asarray(stats_iqr,np.float32)
        self.dataset=str(dataset)
        self.horizon=int(horizon)
        self.training_channels=np.asarray(training_channels,np.int64)
        self.training_query_anchors=np.asarray(training_query_anchors,np.int64)
        self.rerank_best_epoch=int(rerank_best_epoch)

    def scale_stats(self,x):
        return np.clip(
            (x-self.stats_med)/self.stats_iqr,
            -cfg.feature_clip,
            cfg.feature_clip
        ).astype(np.float32)

def safe_torch_load(path):
    try:
        return torch.load(path,map_location=DEVICE,weights_only=False)
    except TypeError:
        return torch.load(path,map_location=DEVICE)

def infer_stack_identity_from_path(path):
    path = Path(path)
    s = str(path)

    dataset = None
    for d in DATASETS:
        if d.lower() in s.lower():
            dataset = d
            break

    m = re.search(r"_H(96|192|336|720)(?:_|\\.|$)", path.name, re.I)
    horizon = int(m.group(1)) if m else None

    return dataset, horizon

def load_stack(path, expected_dataset=None, expected_horizon=None):
    obj = safe_torch_load(path)

    # Only these keys are required for inference.
    required_inference = {
        "query_encoder",
        "history_encoder",
        "reranker",
        "stats_med",
        "stats_iqr",
        "rerank_best_epoch",
    }

    missing = required_inference - set(obj)
    if missing:
        raise KeyError(
            f"{path}: missing inference-critical keys {sorted(missing)}"
        )

    # Older Experiment-11 ETTh1 stacks predate the richer metadata schema.
    # Those missing fields are NOT required by retrieval inference.
    path_dataset, path_horizon = infer_stack_identity_from_path(path)

    dataset = obj.get(
        "dataset",
        expected_dataset if expected_dataset is not None else path_dataset
    )

    horizon = obj.get(
        "horizon",
        expected_horizon if expected_horizon is not None else path_horizon
    )

    if dataset is None:
        raise KeyError(f"{path}: could not infer dataset identity")

    if horizon is None:
        raise KeyError(f"{path}: could not infer horizon identity")

    training_channels = np.asarray(
        obj.get("training_channels", []),
        dtype=np.int64,
    )

    training_query_anchors = np.asarray(
        obj.get("training_query_anchors", []),
        dtype=np.int64,
    )

    seed_everything(cfg.seed)

    qenc = TemporalEncoder1D(
        cfg.enc_hidden,
        cfg.emb_dim,
    ).to(DEVICE)

    henc = TemporalEncoder1D(
        cfg.enc_hidden,
        cfg.emb_dim,
    ).to(DEVICE)

    reranker = WideFusionReranker().to(DEVICE)

    qenc.load_state_dict(
        obj["query_encoder"],
        strict=True,
    )

    henc.load_state_dict(
        obj["history_encoder"],
        strict=True,
    )

    reranker.load_state_dict(
        obj["reranker"],
        strict=True,
    )

    return UpgradedStack(
        qenc,
        henc,
        reranker,
        obj["stats_med"],
        obj["stats_iqr"],
        dataset,
        int(horizon),
        training_channels,
        training_query_anchors,
        obj["rerank_best_epoch"],
    )

def topk_np(scores,k):
    k=min(int(k),len(scores))
    part=np.argpartition(-scores,k-1)[:k]
    return part[np.argsort(-scores[part])]

print("[PASS] PRR implementation loaded")


[PASS] PRR implementation loaded


In [5]:
# Schema diagnostic — older ETTh1 stacks may omit non-inference metadata

schema_rows = []

for d in DATASETS:
    for h in HORIZONS:
        p = STACKS[(d,h)]
        obj = safe_torch_load(p)

        schema_rows.append({
            "dataset": d,
            "horizon": h,
            "n_keys": len(obj) if isinstance(obj, dict) else np.nan,
            "has_dataset": isinstance(obj, dict) and "dataset" in obj,
            "has_horizon": isinstance(obj, dict) and "horizon" in obj,
            "has_training_channels": isinstance(obj, dict) and "training_channels" in obj,
            "has_training_query_anchors": isinstance(obj, dict) and "training_query_anchors" in obj,
            "has_query_encoder": isinstance(obj, dict) and "query_encoder" in obj,
            "has_history_encoder": isinstance(obj, dict) and "history_encoder" in obj,
            "has_reranker": isinstance(obj, dict) and "reranker" in obj,
            "path": str(p),
        })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

critical = (
    schema_df.has_query_encoder
    & schema_df.has_history_encoder
    & schema_df.has_reranker
)

assert critical.all()

print("[PASS] 24/24 stacks contain inference-critical model weights.")
print("[INFO] Missing legacy metadata is allowed and reconstructed from the experiment condition/path.")


,dataset,horizon,n_keys,has_dataset,has_horizon,has_training_channels,has_training_query_anchors,has_query_encoder,has_history_encoder,has_reranker,path
0,ETTh1,96,10,False,True,False,False,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
1,ETTh1,192,10,False,True,False,False,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
2,ETTh1,336,10,False,True,False,False,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
3,ETTh1,720,10,False,True,False,False,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
4,Weather,96,11,True,True,True,True,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
5,Weather,192,11,True,True,True,True,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
6,Weather,336,11,True,True,True,True,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
7,Weather,720,11,True,True,True,True,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
8,Electricity,96,11,True,True,True,True,True,True,True,/data/code/which_histories_matter_ext/checkpoi...
9,Electricity,192,11,True,True,True,True,True,True,True,/data/code/which_histories_matter_ext/checkpoi...


[PASS] 24/24 stacks contain inference-critical model weights.
[INFO] Missing legacy metadata is allowed and reconstructed from the experiment condition/path.


In [6]:

# Cell 5 — verify all 24 stacks strict-load and identity

audit=[]

for d in DATASETS:
    for h in HORIZONS:
        t0=time.perf_counter()
        stack=load_stack(STACKS[(d,h)], expected_dataset=d, expected_horizon=h)

        ok_dataset=(stack.dataset.lower()==d.lower())
        ok_h=(stack.horizon==h)

        audit.append({
            "dataset":d,
            "horizon":h,
            "stack_dataset":stack.dataset,
            "stack_horizon":stack.horizon,
            "training_channels":len(stack.training_channels),
            "training_queries":len(stack.training_query_anchors),
            "rerank_best_epoch":stack.rerank_best_epoch,
            "identity_ok":ok_dataset and ok_h,
            "load_seconds":time.perf_counter()-t0,
        })

        del stack
        gc.collect()
        torch.cuda.empty_cache()

audit_df=pd.DataFrame(audit)
display(audit_df)

assert audit_df.identity_ok.all()
print("[PASS] 24/24 PRR stacks strict-loaded with correct identities")


,dataset,horizon,stack_dataset,stack_horizon,training_channels,training_queries,rerank_best_epoch,identity_ok,load_seconds
0,ETTh1,96,ETTh1,96,0,0,12,True,0.034835
1,ETTh1,192,ETTh1,192,0,0,18,True,0.010025
2,ETTh1,336,ETTh1,336,0,0,7,True,0.011437
3,ETTh1,720,ETTh1,720,0,0,14,True,0.010042
4,Weather,96,Weather,96,21,285,23,True,0.010021
5,Weather,192,Weather,192,21,285,8,True,0.009965
6,Weather,336,Weather,336,21,285,12,True,0.009950
7,Weather,720,Weather,720,21,285,19,True,0.009890
8,Electricity,96,Electricity,96,64,93,14,True,0.010099
9,Electricity,192,Electricity,192,64,93,27,True,0.009966


[PASS] 24/24 PRR stacks strict-loaded with correct identities


In [7]:

# Cell 6 — direct cache + manifest loaders

def load_manifest(d,h,split):
    p=MANIFEST_ROOT/f"{d}_H{h}_{split}.npz"
    assert p.exists(),p
    with np.load(p,allow_pickle=False) as z:
        return (
            np.asarray(z["anchor"],dtype=np.int64),
            np.asarray(z["channel"],dtype=np.int64),
        )

def load_direct(d,h,split):
    p=DIRECT_ROOT/f"{d}_H{h}_{split}_patchtst_direct.npz"
    assert p.exists(),p

    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64)
        c=np.asarray(z["channel"],np.int64)
        direct=np.asarray(z["direct"],np.float32)
        truth=np.asarray(z["truth"],np.float32)

    ma,mc=load_manifest(d,h,split)

    assert np.array_equal(a,ma)
    assert np.array_equal(c,mc)
    assert direct.shape==truth.shape==(len(a),h)

    return a,c,direct,truth

for dataset in DATASETS:
    for h in HORIZONS:
        for split in ("val","test"):
            a, c, direct_arr, truth_arr = load_direct(
                dataset,
                h,
                split,
            )

print("[PASS] Exp34R direct caches align exactly with all Exp33 manifests")


[PASS] Exp34R direct caches align exactly with all Exp33 manifests


In [8]:
# Variable-shadowing guard
# Dataset identifiers must remain strings before the expensive retrieval pass.

for dataset in DATASETS:
    assert isinstance(dataset, str)
    assert dataset in CHANNELS

print("[PASS] dataset identifiers are intact; no shadowing before retrieval.")


[PASS] dataset identifiers are intact; no shadowing before retrieval.


In [9]:

# Cell 7 — per-channel retrieval inference with resumable channel parts

def memory_for_channel(z,ch,h,stack,boundary):
    memory_anchors=np.arange(
        cfg.retrieval_seq_len,
        int(boundary)-int(h)+1,
        cfg.memory_stride,
        dtype=np.int64,
    )

    if len(memory_anchors)<cfg.top_m_pattern:
        raise ValueError(
            f"insufficient memory ch={ch} H={h}: {len(memory_anchors)}"
        )

    pi=(
        memory_anchors[:,None]
        -cfg.retrieval_seq_len
        +np.arange(cfg.retrieval_seq_len)[None,:]
    )

    past=z[pi,ch].astype(np.float32)

    return {
        "anchors":memory_anchors,
        "pattern":batch_pattern(past),
        "stats":stack.scale_stats(context7(past)),
        "emb":encode_rows(stack.henc,past),
    }

@torch.no_grad()
def retrieve_channel(
    z,h,stack,boundary,
    query_anchor,ch,
    query_direct_abs,
    query_truth_abs,
):
    B=len(query_anchor)
    mem=memory_for_channel(z,ch,h,stack,boundary)

    pi=(
        query_anchor[:,None]
        -cfg.retrieval_seq_len
        +np.arange(cfg.retrieval_seq_len)[None,:]
    )
    past=z[pi,ch].astype(np.float32)

    qp=batch_pattern(past)
    qs=stack.scale_stats(context7(past))
    qe=encode_rows(stack.qenc,past)

    ps=np.einsum("bl,ml->bm",qp,mem["pattern"]).astype(np.float32)
    ls=np.einsum("be,me->bm",qe,mem["emb"]).astype(np.float32)

    retrieval_abs=np.empty((B,h),np.float32)
    analog_mse=np.empty(B,np.float32)
    union_size=np.empty(B,np.int16)
    learned_only=np.empty(B,np.int16)

    for left in range(0,B,cfg.query_batch):
        right=min(B,left+cfg.query_batch)
        bsz=right-left

        cand_idx=np.zeros((bsz,cfg.union_max),np.int64)
        valid=np.zeros((bsz,cfg.union_max),bool)

        psl=ps[left:right]
        lsl=ls[left:right]

        for j in range(bsz):
            ptop=topk_np(psl[j],cfg.top_m_pattern)
            ltop=topk_np(lsl[j],cfg.top_m_learned)

            union=np.union1d(ptop,ltop)

            union_size[left+j]=len(union)
            learned_only[left+j]=len(np.setdiff1d(ltop,ptop))

            if len(union)>cfg.union_max:
                op=np.argsort(-psl[j])
                ol=np.argsort(-lsl[j])
                rp=np.empty_like(op); rl=np.empty_like(ol)
                rp[op]=np.arange(1,len(op)+1)
                rl[ol]=np.arange(1,len(ol)+1)
                rrf=1./(60+rp)+1./(60+rl)
                union=union[np.argsort(-rrf[union])[:cfg.union_max]]

            n=len(union)
            cand_idx[j,:n]=union
            valid[j,:n]=True

        base_pattern=np.take_along_axis(
            psl,cand_idx,axis=1
        ).astype(np.float32)

        mstats=mem["stats"][cand_idx]
        memb=mem["emb"][cand_idx]

        qjoint=np.concatenate(
            [qs[left:right],qe[left:right]],
            axis=1,
        )
        cjoint=np.concatenate([mstats,memb],axis=2)
        qrep=np.repeat(qjoint[:,None,:],cfg.union_max,axis=1)

        feat=np.concatenate(
            [
                base_pattern[:,:,None],
                qrep,
                cjoint,
                qrep-cjoint,
                np.abs(qrep-cjoint),
            ],
            axis=2,
        ).astype(np.float32)

        score=stack.reranker(
            torch.from_numpy(feat).float().to(DEVICE),
            torch.from_numpy(base_pattern).float().to(DEVICE),
            torch.from_numpy(valid).to(DEVICE),
        ).cpu().numpy().astype(np.float32)

        top=np.argsort(-score,axis=1)[:,:cfg.top_k]
        selected_idx=np.take_along_axis(cand_idx,top,axis=1)
        selected_anchor=mem["anchors"][selected_idx]

        offsets=np.arange(h,dtype=np.int64)
        future=z[
            selected_anchor[:,:,None]+offsets[None,None,:],
            ch,
        ].astype(np.float32)

        current=z[selected_anchor-1,ch].astype(np.float32)
        cand_resid=future-current[:,:,None]

        qcur=z[query_anchor[left:right]-1,ch].astype(np.float32)
        retrieved_resid=cand_resid.mean(axis=1)
        retrieval_abs[left:right]=retrieved_resid+qcur[:,None]

        true_resid=(
            query_truth_abs[left:right]
            -qcur[:,None]
        )
        analog_mse[left:right]=(
            (cand_resid-true_resid[:,None,:])**2
        ).mean(axis=(1,2))

    return {
        "retrieval":retrieval_abs,
        "analog_mse":analog_mse,
        "union_size":union_size,
        "learned_only":learned_only,
    }

def part_path(d,h,split,ch):
    p=PART_ROOT/d/f"H{h}"/split
    p.mkdir(parents=True,exist_ok=True)
    return p/f"ch{ch:04d}.npz"

def part_valid(path,anchors,h):
    if not path.exists():
        return False
    try:
        with np.load(path,allow_pickle=False) as z:
            return (
                np.array_equal(z["anchor"],anchors)
                and z["retrieval"].shape==(len(anchors),h)
                and z["analog_mse"].shape==(len(anchors),)
            )
    except Exception:
        return False

print("[PASS] retrieval inference functions ready")


[PASS] retrieval inference functions ready


## Frozen 24-condition evaluation

Only the **test** split is needed for the Table-1 fairness control.  
The checkpoint, normalization, memory stride, query manifests, target futures, and reranker are
unchanged from full PRR. The only change is the candidate support cap.


In [10]:
# Run the 24 test conditions, resumably by channel.

summary_rows = []

for d in DATASETS:
    z = DATA[d]["z"]

    for h in HORIZONS:
        print("\n" + "="*110)
        print(f"PRR-B100 | {d} | H={h}")
        print("="*110)

        stack = load_stack(STACKS[(d,h)], expected_dataset=d, expected_horizon=h)
        assert stack.dataset.lower() == d.lower()
        assert stack.horizon == h

        a, c, direct, truth = load_direct(d, h, "test")
        boundary = DATA[d]["val_end"]

        analog = np.empty(len(a), np.float32)
        union_sz = np.empty(len(a), np.int16)
        learned_only = np.empty(len(a), np.int16)

        used_channels = np.unique(c)
        t0 = time.perf_counter()

        for ci, ch in enumerate(used_channels):
            rows = np.flatnonzero(c == ch)
            aa = a[rows]
            pp = part_path(d, h, "test", int(ch))

            if part_valid(pp, aa, h):
                with np.load(pp, allow_pickle=False) as zz:
                    analog[rows] = zz["analog_mse"]
                    union_sz[rows] = zz["union_size"]
                    learned_only[rows] = zz["learned_only"]
            else:
                out = retrieve_channel(
                    z, h, stack, boundary,
                    aa, int(ch),
                    direct[rows],
                    truth[rows],
                )

                analog[rows] = out["analog_mse"]
                union_sz[rows] = out["union_size"]
                learned_only[rows] = out["learned_only"]

                tmp = pp.with_suffix(".tmp.npz")
                np.savez_compressed(
                    tmp,
                    anchor=aa,
                    retrieval=out["retrieval"],
                    analog_mse=out["analog_mse"],
                    union_size=out["union_size"],
                    learned_only=out["learned_only"],
                )
                os.replace(tmp, pp)

            if ci == 0 or (ci+1) % 100 == 0 or ci+1 == len(used_channels):
                print(f"  channel {ci+1}/{len(used_channels)}")

        sec = time.perf_counter() - t0

        assert int(union_sz.max()) <= 100, (d, h, union_sz.max())

        row = {
            "dataset": d,
            "horizon": int(h),
            "prr_budget100_analog_future_mse": float(analog.mean()),
            "mean_union_size": float(union_sz.mean()),
            "median_union_size": float(np.median(union_sz)),
            "max_union_size": int(union_sz.max()),
            "mean_learned_only_candidates": float(learned_only.mean()),
            "test_pairs": int(len(a)),
            "seconds": float(sec),
        }
        summary_rows.append(row)

        print(
            f"  AnalogFutureMSE={row['prr_budget100_analog_future_mse']:.6f} | "
            f"mean union={row['mean_union_size']:.2f} | "
            f"max union={row['max_union_size']} | "
            f"learned-only={row['mean_learned_only_candidates']:.2f}"
        )

        del stack
        gc.collect()
        torch.cuda.empty_cache()

budget_df = pd.DataFrame(summary_rows).sort_values(["dataset","horizon"]).reset_index(drop=True)
assert len(budget_df) == 24
assert budget_df["max_union_size"].max() <= 100

display(budget_df)
budget_df.to_csv(RESULT_ROOT / "prr_budget100_test_24.csv", index=False)

print("[PASS] 24/24 PRR-B100 conditions complete")



PRR-B100 | ETTh1 | H=96
  channel 1/7
  channel 7/7
  AnalogFutureMSE=1.169867 | mean union=95.80 | max union=100 | learned-only=45.80

PRR-B100 | ETTh1 | H=192
  channel 1/7
  channel 7/7
  AnalogFutureMSE=1.223163 | mean union=96.04 | max union=100 | learned-only=46.04

PRR-B100 | ETTh1 | H=336
  channel 1/7
  channel 7/7
  AnalogFutureMSE=1.480429 | mean union=96.52 | max union=100 | learned-only=46.52

PRR-B100 | ETTh1 | H=720
  channel 1/7
  channel 7/7
  AnalogFutureMSE=1.467654 | mean union=97.03 | max union=100 | learned-only=47.03

PRR-B100 | Weather | H=96
  channel 1/21
  channel 21/21
  AnalogFutureMSE=0.421212 | mean union=98.10 | max union=100 | learned-only=48.10

PRR-B100 | Weather | H=192
  channel 1/21
  channel 21/21
  AnalogFutureMSE=0.533346 | mean union=97.71 | max union=100 | learned-only=47.71

PRR-B100 | Weather | H=336
  channel 1/21
  channel 21/21
  AnalogFutureMSE=0.688207 | mean union=98.05 | max union=100 | learned-only=48.05

PRR-B100 | Weather | H=720


,dataset,horizon,prr_budget100_analog_future_mse,mean_union_size,median_union_size,max_union_size,mean_learned_only_candidates,test_pairs,seconds
0,ETTh1,96,1.169867,95.804443,97.0,100,45.804443,8192,2.560405
1,ETTh1,192,1.223163,96.036987,97.0,100,46.036987,8192,2.328950
2,ETTh1,336,1.480429,96.524414,98.0,100,46.524414,8192,2.572823
3,ETTh1,720,1.467654,97.030029,98.0,100,47.030029,8192,3.356693
4,Electricity,96,0.721195,94.115845,96.0,100,44.115845,8192,5.610051
5,Electricity,192,0.735946,94.174072,96.0,100,44.174072,8192,5.657362
6,Electricity,336,0.767276,94.168457,96.0,100,44.168457,8192,5.867614
7,Electricity,720,0.863012,95.382080,97.0,100,45.382080,8192,6.503285
8,Exchange,96,0.172511,96.864624,97.0,100,46.864624,8192,1.839873
9,Exchange,192,0.402023,96.404785,97.0,100,46.404785,8192,2.045558


[PASS] 24/24 PRR-B100 conditions complete


In [11]:
# Load the submitted Pearson / PRR-Stat / full-PRR reference results.
# Prefer project files; fall back to the exact submitted 24-row values.

reference_candidates = [
    ROOT / "results" / "retrieval_24_conditions.csv",
    ROOT / "results" / "stride8_revalidation" / "retrieval_24_conditions.csv",
    ROOT / "results" / "stride8_revalidation" / "table1_stride8_revalidated.csv",
]

ref_path = next((p for p in reference_candidates if p.exists()), None)

if ref_path is not None:
    ref0 = pd.read_csv(ref_path)
    rename = {
        "pearson_analog_future_mse": "pearson",
        "prr_stat_analog_future_mse": "prr_stat",
        "prr_analog_future_mse": "prr",
    }
    ref0 = ref0.rename(columns=rename)

    needed = {"dataset","horizon","pearson","prr"}
    if "prr_stat" not in ref0.columns:
        print("[INFO] project reference lacks PRR-Stat; using exact submitted fallback for that column.")
        ref0 = None
    elif not needed.issubset(ref0.columns):
        ref0 = None
else:
    ref0 = None

if ref0 is None:
    submitted = """dataset,horizon,pearson,prr_stat,prr
ETTh1,96,1.612267,1.1965,1.1643
ETTh1,192,1.817531,1.3664,1.2229
ETTh1,336,1.992872,1.4861,1.4816
ETTh1,720,2.254092,1.7272,1.4618
Weather,96,1.161329,0.670706,0.4316
Weather,192,1.062809,0.733725,0.5147
Weather,336,1.290959,0.912211,0.6748
Weather,720,1.295817,1.167582,0.8130
Electricity,96,1.179589,0.875108,0.7191
Electricity,192,1.195698,0.896645,0.7345
Electricity,336,1.241844,0.930136,0.7648
Electricity,720,1.287729,0.976864,0.8602
Traffic,96,1.973787,1.475479,1.4696
Traffic,192,2.004524,1.513514,1.4508
Traffic,336,2.063635,1.578335,1.5147
Traffic,720,2.166454,1.757110,1.7351
Exchange,96,0.213266,0.220600,0.1652
Exchange,192,0.423408,0.427600,0.4116
Exchange,336,0.764282,0.771400,0.8094
Exchange,720,1.886809,2.131000,1.7770
Solar,96,0.588674,0.526962,0.5087
Solar,192,0.597890,0.541064,0.5406
Solar,336,0.623836,0.569167,0.5701
Solar,720,0.638530,0.578178,0.5507
"""
    from io import StringIO
    ref = pd.read_csv(StringIO(submitted))
    ref_source = "exact submitted fallback"
else:
    ref = ref0[["dataset","horizon","pearson","prr_stat","prr"]].copy()
    ref_source = str(ref_path)

ref["horizon"] = ref["horizon"].astype(int)
assert len(ref) == 24
print("Reference source:", ref_source)
display(ref.head())


Reference source: /data/code/which_histories_matter_ext/results/stride8_revalidation/table1_stride8_revalidated.csv


,dataset,horizon,pearson,prr_stat,prr
0,ETTh1,96,1.612267,1.196500,1.1643
1,ETTh1,192,1.817531,1.366400,1.2229
2,ETTh1,336,1.992872,1.486100,1.4816
3,ETTh1,720,2.254092,1.727200,1.4618
4,Electricity,96,1.179589,0.875108,0.7191


In [12]:
# Matched comparison and paper-facing statistics.

comp = ref.merge(
    budget_df[[
        "dataset","horizon",
        "prr_budget100_analog_future_mse",
        "mean_union_size","median_union_size","max_union_size",
        "mean_learned_only_candidates",
    ]],
    on=["dataset","horizon"],
    how="inner",
)
comp = comp.rename(columns={"prr_budget100_analog_future_mse":"prr_budget100"})
assert len(comp) == 24

def gain_pct(method, baseline):
    return 100.0 * (comp[baseline] - comp[method]) / comp[baseline]

def wtl(g, tol=1e-12):
    g = np.asarray(g, float)
    return {
        "W": int((g > tol).sum()),
        "T": int((np.abs(g) <= tol).sum()),
        "L": int((g < -tol).sum()),
        "mean_gain_pct": float(g.mean()),
        "median_gain_pct": float(np.median(g)),
    }

comparisons = []
for base, label in [
    ("pearson","Pearson"),
    ("prr_stat","PRR-Stat"),
    ("prr","Full PRR"),
]:
    g = gain_pct("prr_budget100", base)
    comparisons.append({
        "comparison": f"PRR-B100 vs {label}",
        **wtl(g),
    })

summary = pd.DataFrame(comparisons)

comp["budget100_gain_vs_pearson_pct"] = gain_pct("prr_budget100","pearson")
comp["budget100_gain_vs_stat_pct"] = gain_pct("prr_budget100","prr_stat")
comp["full_prr_gain_vs_budget100_pct"] = 100.0 * (comp["prr_budget100"] - comp["prr"]) / comp["prr_budget100"]

display(comp[[
    "dataset","horizon","pearson","prr_stat","prr_budget100","prr",
    "budget100_gain_vs_pearson_pct","mean_union_size","max_union_size"
]])
display(summary)

comp.to_csv(RESULT_ROOT / "prr_budget100_exact_24.csv", index=False)
summary.to_csv(RESULT_ROOT / "prr_budget100_summary.csv", index=False)

print("\n" + "="*112)
print("PAPER-FACING PRR-BUDGET100 RESULT")
print("="*112)
print("Definition: same frozen full-PRR model; inference candidates = Pearson Top50 ∪ learned Top50; union <=100.")
print(f"Mean union size:   {comp['mean_union_size'].mean():.2f}")
print(f"Median union size: {comp['median_union_size'].median():.2f}")
print(f"Maximum union:     {comp['max_union_size'].max():.0f}")

for r in summary.itertuples(index=False):
    print(
        f"{r.comparison:34s}: "
        f"{r.W}W/{r.T}T/{r.L}L | "
        f"mean={r.mean_gain_pct:+.3f}% | median={r.median_gain_pct:+.3f}%"
    )

print("\nPer-dataset PRR-B100 vs Pearson:")
for d, gdf in comp.groupby("dataset", sort=False):
    gains = 100.0 * (gdf["pearson"] - gdf["prr_budget100"]) / gdf["pearson"]
    s = wtl(gains)
    print(
        f"{d:11s}: {s['W']}W/{s['T']}T/{s['L']}L | "
        f"mean={s['mean_gain_pct']:+.3f}%"
    )

print("\nSaved:")
for p in sorted(RESULT_ROOT.glob("prr_budget100*.csv")):
    print(" -", p)


,dataset,horizon,pearson,prr_stat,prr_budget100,prr,budget100_gain_vs_pearson_pct,mean_union_size,max_union_size
0,ETTh1,96,1.612267,1.196500,1.169867,1.1643,27.439652,95.804443,100
1,ETTh1,192,1.817531,1.366400,1.223163,1.2229,32.701947,96.036987,100
2,ETTh1,336,1.992872,1.486100,1.480429,1.4816,25.713782,96.524414,100
3,ETTh1,720,2.254092,1.727200,1.467654,1.4618,34.889376,97.030029,100
4,Electricity,96,1.179589,0.875108,0.721195,0.7191,38.860475,94.115845,100
5,Electricity,192,1.195698,0.896645,0.735946,0.7345,38.450531,94.174072,100
6,Electricity,336,1.241844,0.930136,0.767276,0.7648,38.214742,94.168457,100
7,Electricity,720,1.287729,0.976864,0.863012,0.8602,32.981870,95.382080,100
8,Exchange,96,0.213266,0.220600,0.172511,0.1652,19.110231,96.864624,100
9,Exchange,192,0.423408,0.427600,0.402023,0.4116,5.050690,96.404785,100


,comparison,W,T,L,mean_gain_pct,median_gain_pct
0,PRR-B100 vs Pearson,23,0,1,26.555823,27.047751
1,PRR-B100 vs PRR-Stat,22,0,2,11.316042,8.232143
2,PRR-B100 vs Full PRR,11,0,13,-0.250801,-0.088803



PAPER-FACING PRR-BUDGET100 RESULT
Definition: same frozen full-PRR model; inference candidates = Pearson Top50 ∪ learned Top50; union <=100.
Mean union size:   94.93
Median union size: 97.00
Maximum union:     100
PRR-B100 vs Pearson          : 23W/0T/1L | mean=+26.556% | median=+27.048%
PRR-B100 vs PRR-Stat         : 22W/0T/2L | mean=+11.316% | median=+8.232%
PRR-B100 vs Full PRR         : 11W/0T/13L | mean=-0.251% | median=-0.089%

Per-dataset PRR-B100 vs Pearson:
ETTh1      : 4W/0T/0L | mean=+30.186%
Electricity: 4W/0T/0L | mean=+37.127%
Exchange   : 3W/0T/1L | mean=+5.984%
Solar      : 4W/0T/0L | mean=+11.652%
Traffic    : 4W/0T/0L | mean=+25.451%
Weather    : 4W/0T/0L | mean=+48.934%

Saved:
 - /data/code/which_histories_matter_ext/results/exp49_prr_budget100/prr_budget100_exact_24.csv
 - /data/code/which_histories_matter_ext/results/exp49_prr_budget100/prr_budget100_summary.csv
 - /data/code/which_histories_matter_ext/results/exp49_prr_budget100/prr_budget100_test_24.csv


## Decision rule for the paper

Do **not** decide before seeing the result.

A strong outcome would be:

- PRR-B100 still improves Pearson in a clear majority of the 24 conditions;
- the mean gain remains substantial rather than collapsing toward zero;
- the average union is indeed below 100;
- its performance is competitive with, or better than, PRR-Stat.

If that holds, the cleanest main-table layout is:

`Dataset | H | Pearson | PRR-Stat | PRR-B100 | PRR (ours)`

and the caption should state:

> PRR-Stat reranks the same Pearson Top-100 support. PRR-B100 uses the same frozen full-PRR
> model but caps inference support at Pearson Top-50 ∪ learned Top-50 (≤100 unique candidates).
> Full PRR uses Top-100 ∪ Top-100 (≈183 unique candidates on average).

This directly separates three questions:

1. Does future-supervised relevance help within fixed Pearson support? → **PRR-Stat**
2. Does learned support still help at a matched ≤100 candidate budget? → **PRR-B100**
3. What is the performance of the full proposed retrieval policy? → **Full PRR**
